# 🥚 Pelatihan YOLOv8-seg (Instance Segmentation) - Cacat Cangkang Telur\nNotebook ini melatih model **YOLOv8-seg** untuk melakukan **segmentasi piksel cacat retak (*cracked*) dan noda kotoran (*dirt*)** pada cangkang telur ayam sesuai acuan standar SNI 3926:2023.\n\n### 📊 Distribusi & Split Dataset (70% Train / 20% Valid / 10% Test):\n- **Total Dataset:** 892 gambar (1.166 polygon mask cacat)\n  - **Train (70%):** 624 gambar (207 cracked, 225 dirt, 192 good)\n  - **Valid (20%):** 178 gambar (59 cracked, 64 dirt, 55 good)\n  - **Test (10%):** 90 gambar (30 cracked, 32 dirt, 28 good)\n\n### 🏷️ Pembagian Kelas Mutu SNI:\n1. cracked (296 gambar, 330 polygon instances): Cacat kritis retak kerabang (Reject / Mutu III).\n2. dirt (321 gambar, 836 polygon instances): Noda kotoran / feses permukaan (Mutu II / III).\n3. good (275 gambar, label kosong): Sampel negatif background cangkang bersih mulus (Mutu I).\n\n### ⚙️ Konfigurasi Pelatihan:\n- **Arsitektur:** yolov8n-seg.pt (Instance Segmentation)\n- **Epochs:** 80 (Patience Early Stopping: 15)\n- **Input Resolution:** 640×640\n- **Augmentasi Spesifik:** mosaic=0.5, close_mosaic=10, hsv_v=0.4, hsv_s=0.4\n

In [ ]:
# 1. Cek Akselerasi GPU Colab & Install Ultralytics
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# 2. Persiapan & Ekstraksi Dataset Segmentasi
import os
import zipfile
import subprocess

extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)

# Cari file zip di folder /content
zip_candidates = [f for f in os.listdir('/content') if f.endswith('.zip') and not f.startswith('.')]

if not zip_candidates:
    print('ℹ️ Belum ada file .zip di /content.')
    print("👉 Silakan upload file 'Egg_Dataset_Clean_70_20_10.zip' ke panel Files (kiri Colab).")
    print('👉 PERHATIKAN: Tunggu sampai lingkaran orange progress upload di pojok kiri bawah selesai penuh!')
else:
    zip_path = os.path.join('/content', zip_candidates[0])
    file_size_bytes = os.path.getsize(zip_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f'📦 File terdeteksi : {zip_path}')
    print(f'💾 Ukuran file     : {file_size_mb:.2f} MB ({file_size_bytes:,} bytes)')
    
    # Validasi: Ukuran asli dataset adalah ~23.5 MB
    if file_size_mb < 5.0:
        print('\n⚠️ PERINGATAN: Ukuran file zip terlalu kecil (< 5 MB)!')
        print('Penyebab error: File zip belum selesai di-upload 100% atau koneksi sempat terputus.')
        print('Cara mengatasi:')
        print('  1. Klik kanan file zip di panel Files kiri Colab -> Delete file.')
        print('  2. Upload ulang file Egg_Dataset_Clean_70_20_10.zip.')
        print('  3. Tunggu lingkaran progress di kiri bawah sampai 100% selesai, baru jalankan cell ini!')
    else:
        print(f'\n🚀 Mengekstrak ke {extract_dir}...')
        # Ekstraksi menggunakan perintah Linux unzip bawaan Colab (cepat & handal)
        res = subprocess.run(['unzip', '-q', '-o', zip_path, '-d', extract_dir], capture_output=True, text=True)
        if res.returncode == 0:
            print('✅ Ekstraksi dataset berhasil!')
        else:
            print(f'Peringatan command unzip: {res.stderr}. Mencoba modul zipfile...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print('✅ Ekstraksi via zipfile berhasil!')
        
        # Verifikasi folder dataset
        print(f'📂 Isi folder {extract_dir}: {os.listdir(extract_dir)}')


In [ ]:
# 3. Konfigurasi data.yaml (Path Dataset Colab)
import os
import yaml

# Cari file data.yaml di dalam folder ekstraksi
found_yaml = None
for root, dirs, files in os.walk('/content'):
    if 'data.yaml' in files:
        found_yaml = os.path.join(root, 'data.yaml')
        break

if found_yaml:
    with open(found_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    # Set absolute root path ke direktori tempat data.yaml berada
    cfg['path'] = os.path.dirname(found_yaml)
    with open(found_yaml, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    data_yaml_path = found_yaml
    print(f'✅ Konfigurasi data.yaml ditemukan di: {data_yaml_path}')
    print(yaml.dump(cfg))
else:
    data_yaml_path = '/content/dataset/data.yaml'
    default_cfg = {
        'path': '/content/dataset',
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 2,
        'names': ['cracked', 'dirt']
    }
    os.makedirs('/content/dataset', exist_ok=True)
    with open(data_yaml_path, 'w') as f:
        yaml.dump(default_cfg, f, default_flow_style=False)
    print(f'✅ data.yaml baru berhasil dibuat di: {data_yaml_path}')
    print(yaml.dump(default_cfg))

In [ ]:
# 4. Mulai Pelatihan Model YOLOv8-seg (Instance Segmentation)
from ultralytics import YOLO
import torch

# Deteksi perangkat otomatis (GPU jika ada, otomatis CPU jika tanpa GPU)
train_device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 16 if torch.cuda.is_available() else 8
epochs_num = 80  # Ditingkatkan dari 35 ke 80 agar konvergensi fitur fraktur lebih matang

print(f'🚀 Memulai pelatihan YOLOv8-seg pada perangkat: {train_device} (Batch: {batch_size}, Epochs: {epochs_num})...')

# Memuat arsitektur pre-trained YOLOv8 Nano Segmentation (Ringan & Cepat untuk Konveyor)
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data=data_yaml_path,
    epochs=epochs_num,
    imgsz=640,
    batch=batch_size,
    device=train_device,
    project='/content/runs',
    name='egg_defect_yolov8_seg',
    exist_ok=True,
    patience=15,         # Early stopping jika 15 epoch berturut-turut mAP tidak naik
    save=True,
    plots=True,
    verbose=True,
    # Parameter augmentasi terarah khusus retakan rambut & cangkang telur:
    mosaic=0.5,          # Mengurangi mosaic agar garis fraktur tipis tidak terpotong acak
    close_mosaic=10,     # Mematikan mosaic 10 epoch terakhir untuk penyempurnaan mask cangkang
    hsv_v=0.4,           # Simulasi variasi bayangan dan pencahayaan lampu konveyor
    hsv_s=0.4,           # Simulasi variasi spektrum warna cangkang cokelat ras
    fliplr=0.5,          # Flip horizontal 50%
    flipud=0.5           # Flip vertikal 50%
)

In [ ]:
# 5. Evaluasi Performa Model (Box mAP & Mask mAP)
metrics = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='eval_test_seg'
)

print('\n' + '='*55)
print('📊 HASIL EVALUASI SEGMENTASI (TEST SET):')
print('='*55)
print(f'📦 Bounding Box mAP50        : {metrics.box.map50:.4f}')
print(f'📦 Bounding Box mAP50-95     : {metrics.box.map:.4f}')
print(f'🎭 Segmentation Mask mAP50   : {metrics.seg.map50:.4f}')
print(f'🎭 Segmentation Mask mAP50-95: {metrics.seg.map:.4f}')
print('='*55)

In [ ]:
# 6. Visualisasi Hasil Segmentasi (Masking Cacat Cangkang)
import os
import glob
from IPython.display import Image, display

# Menampilkan visualisasi hasil prediksi segmentasi
val_preds = glob.glob('/content/runs/egg_defect_yolov8_seg/val_batch*_pred.jpg')
charts = [
    '/content/runs/egg_defect_yolov8_seg/results.png',
    '/content/runs/egg_defect_yolov8_seg/confusion_matrix.png'
]

for path in charts + val_preds[:2]:
    if os.path.exists(path):
        print(f'\n=== {os.path.basename(path)} ===')
        display(Image(path, width=720))

In [ ]:
# 7. Unduh Bobot Model Segmentasi Terbaik (best.pt)
from google.colab import files

best_seg_weight = '/content/runs/egg_defect_yolov8_seg/weights/best.pt'
if os.path.exists(best_seg_weight):
    print('⬇️ Mengunduh file best.pt ke komputer Anda...')
    files.download(best_seg_weight)
else:
    print('⚠️ File best.pt belum ditemukan. Pastikan proses training sudah selesai.')

In [ ]:
# 8. Unduh Seluruh Grafik Pelatihan, Kurva Metrik & Matriks Konfusi
import os
import shutil
import glob
from IPython.display import Image, display
from google.colab import files

# 1. Deteksi otomatis folder hasil pelatihan YOLOv8-seg
run_dir = None
if 'results' in locals() and hasattr(results, 'save_dir'):
    run_dir = str(results.save_dir)
elif os.path.exists('/content/runs/egg_defect_yolov8_seg'):
    run_dir = '/content/runs/egg_defect_yolov8_seg'
else:
    # Cari folder yang memuat file results.png
    candidates = glob.glob('/content/runs/**/results.png', recursive=True)
    if candidates:
        run_dir = os.path.dirname(candidates[0])

if not run_dir or not os.path.exists(run_dir):
    print('⚠️ Folder hasil training tidak ditemukan. Pastikan cell pelatihan (Cell 4) sudah selesai!')
else:
    print(f'📂 Folder hasil training ditemukan: {run_dir}')
    
    # 2. Tampilkan Preview Matriks Konfusi & Grafik Hasil di Colab
    cm_path = os.path.join(run_dir, 'confusion_matrix_normalized.png')
    if not os.path.exists(cm_path):
        cm_path = os.path.join(run_dir, 'confusion_matrix.png')
    
    res_plot_path = os.path.join(run_dir, 'results.png')
    
    if os.path.exists(cm_path):
        print('\n📊 Preview Matriks Konfusi (Normalized):')
        display(Image(filename=cm_path, width=600))
    
    if os.path.exists(res_plot_path):
        print('\n📈 Preview Kurva Loss dan mAP Pelatihan:')
        display(Image(filename=res_plot_path, width=800))
    
    # 3. Kumpulkan seluruh file visual (.png, .jpg, .csv) ke folder khusus
    export_dir = '/content/grafik_dan_matriks_evaluasi'
    if os.path.exists(export_dir):
        shutil.rmtree(export_dir)
    os.makedirs(export_dir, exist_ok=True)
    
    plot_extensions = ('.png', '.jpg', '.jpeg', '.csv')
    copied_count = 0
    for fname in os.listdir(run_dir):
        if fname.lower().endswith(plot_extensions):
            src = os.path.join(run_dir, fname)
            dst = os.path.join(export_dir, fname)
            shutil.copy2(src, dst)
            copied_count += 1
    
    # 4. Kompresi folder grafik menjadi file ZIP
    zip_output_path = '/content/grafik_dan_matriks_evaluasi.zip'
    if os.path.exists(zip_output_path):
        os.remove(zip_output_path)
        
    shutil.make_archive('/content/grafik_dan_matriks_evaluasi', 'zip', export_dir)
    
    print(f'\n📦 Berhasil mengemas {copied_count} file grafik, kurva, matriks, dan rekapan CSV!')
    print("⬇️ Memulai unduhan otomatis: 'grafik_dan_matriks_evaluasi.zip'...")
    files.download(zip_output_path)
